# Hulpcode LLM Sims game
Deze Notebook bevat een heel klein beginnetje voor de portfolio opdracht. 

De package `tkinter` is een standaard package in Python voor het maken van GUI's. Het is een beetje ouderwets, maar het werkt prima voor deze opdracht.

In [7]:
import tkinter as tk
from tkinter import simpledialog

De volgende cell bevat configuratie code voor de GUI. 

In [8]:
GRID_SIZE = 15
CELL_SIZE = 40
DEFAULT_SPEED = 1000

De volgende code bevat geen logica voor het spel, maar alleen de GUI. Dit is voor nu niet zo interessant. 

In [9]:
from abc import abstractmethod

speler = "🙂"

class SimsWereld:
    def __init__(self, root):
        self.root = root
        self.root.title("LLM Sims!")
        self.canvas = tk.Canvas(root, width=GRID_SIZE*CELL_SIZE, height=GRID_SIZE*CELL_SIZE)
        self.canvas.bind("<Button-1>", self.on_canvas_click)
        self.canvas.pack()
        self.speed = DEFAULT_SPEED
        [self.running, self.step_mode] = [False, False]
        self.stopping = False
        def on_close():
            print("Afsluiten...")
            self.stopping = True
            self.root.destroy()
        self.root.protocol("WM_DELETE_WINDOW", on_close)
        self.grid = [[{} for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
        self.initialiseer_wereld()
        self.draw_grid()
        self.control_frame = tk.Frame(root)
        self.control_frame.pack()
        self.create_buttons()
        self.root.after(self.speed, self.update_world)
    def draw_grid(self):
        self.canvas.delete("all")
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                x1, y1 = x * CELL_SIZE, y * CELL_SIZE
                x2, y2 = x1 + CELL_SIZE, y1 + CELL_SIZE
                self.canvas.create_rectangle(x1, y1, x2, y2, fill="white", outline="gray")
                if self.grid[y][x]:
                    self.canvas.create_text(x1 + CELL_SIZE // 2, y1 + CELL_SIZE // 2, text=self.grid[y][x]['icon'], font=("Arial", 16))
    def create_buttons(self):
        def play(): [self.step_mode, self.running] = [False, not self.running]
        tk.Button(self.control_frame, text="▶️ Play/Pauze", command=play).grid(row=0, column=0)
        def stap(): [self.step_mode, self.running] = [True, True]
        tk.Button(self.control_frame, text="⏭️ Stap", command=stap).grid(row=0, column=1)
        def snelheid(): self.speed = simpledialog.askinteger("Snelheid", "Nieuwe snelheid in ms (lager = sneller):", initialvalue=self.speed)
        tk.Button(self.control_frame, text="⏩ Snelheid", command=snelheid).grid(row=0, column=2)
        def mededeling(): 
            instructie = simpledialog.askstring("Verzend instructie naar alle sims", "Geef alle Sims een opdracht (bijv. 'het gaat regenen'):")
            for y in range(GRID_SIZE):
                for x in range(GRID_SIZE):
                    if self.grid[y][x]["type"] == "speler":
                        self.instrueer(self.grid[y][x], instructie)
        tk.Button(self.control_frame, text="📢 Instructie", command=mededeling).grid(row=0, column=3)
    def update_world(self):
        if self.running:
            self.doe_alles()
            self.draw_grid()
            if self.step_mode:
                self.running = False
        if not self.stopping:
            self.root.after(self.speed, self.update_world)
    def on_canvas_click(self, event):
        grid_x = event.x // CELL_SIZE
        grid_y = event.y // CELL_SIZE
        if 0 <= grid_x < GRID_SIZE and 0 <= grid_y < GRID_SIZE:
            if self.grid[grid_y][grid_x]["type"] == "speler":
                self.instrueer(self.grid[grid_y][grid_x], simpledialog.askstring("Instructie", "Geef de Sim een opdracht (bijv. 'ga naar de radio'):"))
            
    @abstractmethod
    def initialiseer_wereld(self):
        pass
    @abstractmethod
    def doe_alles(self):
        pass
    def instrueer(self, sim, instructie):
        sim["instructies"] += [instructie]



Dit zijn de objecten in het voorbeeld: 

In [10]:
objecten = {
    "🌳": "boom",
    "🛌": "bed",
    "🍏": "appel",
    "🎶": "radio"
}

De volgende code bevat de logica van het spel. Aanpassingen zul je hier moeten maken om het spel te laten werken volgens de opdrachtomschrijving. 

In [11]:
import random
random.seed(42)

class MijnSimsWereld(SimsWereld):
    def __init__(self):
        super().__init__(tk.Tk())
        self.root.mainloop()
    
    def initialiseer_wereld(self):
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                if random.random() < 0.1:
                    icon = random.choice(list(objecten.keys()))
                    self.grid[y][x] = { 'type': objecten[icon], 'icon': icon}
                elif random.random() < 0.01:
                    self.grid[y][x] = { 'type': 'speler', 'icon': speler, 'hunger': 10, 'instructies': [] }

    def doe_alles(self):
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                if self.grid[y][x].get('type') == 'speler':
                    self.grid[y][x]['hunger'] -= 1
                    dx, dy = random.choice([(0, 1), (1, 0), (0, -1), (-1, 0)])
                    if self.grid[y][x]['instructies']:
                        instructie = self.grid[y][x]['instructies'].pop(0)
                        if "rennen" in instructie:
                            dx *= 2
                            dy *= 2
                    new_x = max(0, min(GRID_SIZE - 1, x + dx))
                    new_y = max(0, min(GRID_SIZE - 1, y + dy))
                    if (not self.grid[new_y][new_x]) or (self.grid[new_y][new_x].get('type') == 'appel' and self.grid[y][x]['hunger'] < 5):
                        if self.grid[new_y][new_x].get('type') == 'appel':
                            self.grid[y][x]['hunger'] += 5
                        self.grid[new_y][new_x] = self.grid[y][x]
                        self.grid[y][x] = {}
                    for y2 in range(GRID_SIZE):
                        for x2 in range(GRID_SIZE):
                            if self.grid[y2][x2].get('type') == 'speler' and (y2 != y or x2 != x):
                                if random.random() < 0.1:
                                    print(f"Speler op ({x},{y}) zegt hallo tegen speler op ({x2},{y2})")
                                

We starten de game. 

In [12]:
MijnSimsWereld()

Speler op (7,3) zegt hallo tegen speler op (2,13)
Speler op (2,14) zegt hallo tegen speler op (3,14)
Speler op (3,13) zegt hallo tegen speler op (7,4)
Speler op (1,13) zegt hallo tegen speler op (7,4)
Speler op (7,3) zegt hallo tegen speler op (0,14)
Speler op (0,14) zegt hallo tegen speler op (0,13)
Speler op (0,13) zegt hallo tegen speler op (1,13)
Speler op (3,14) zegt hallo tegen speler op (2,14)
Afsluiten...
